# Deutsche Bahn Historical Data Catalog

This notebook creates the first formal **data catalog** for the Deutsche Bahn historical delay dataset used in this project.

A data catalog documents:

- the dataset source and purpose;
- the proposed row grain;
- column names, types, roles, and meanings;
- identifiers and candidate keys;
- timestamp units and time coverage;
- assumptions, risks, and questions that still need verification.

This notebook documents what the dataset is **supposed to mean**.

In [4]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

DATA_PATH = Path(
    "/opt/spark/work-dir/data/historical_delays/data-2024-07.parquet"
)

CATALOG_BASE_PATH = Path(
    "/opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical"
)

print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print(
        "Dataset size (MB):",
        round(DATA_PATH.stat().st_size / (1024**2), 2),
    )

Dataset path: /opt/spark/work-dir/data/historical_delays/data-2024-07.parquet
Dataset exists: True
Dataset size (MB): 101.87


## 2. Reuse or create the Spark session

The Parquet source stores timestamp fields with **nanosecond precision**. Spark 3.5 cannot directly interpret `TIMESTAMP(NANOS, false)` as a Spark timestamp.

The compatibility option below lets Spark read these fields as `long` values. We then convert only the known timestamp columns from nanoseconds to microseconds and finally to Spark timestamps.

The notebook connects to the existing standalone cluster:

```text
spark://spark-master:7077
```

In [23]:
SOURCE_TIME_ZONE = "Europe/Berlin"

try:
    spark
except NameError:
    spark = (
        SparkSession.builder
        .appName("deutsche-bahn-data-catalog")
        .master("spark://spark-master:7077")
        .config("spark.driver.host", "spark-jupyter")
        .config("spark.driver.bindAddress", "0.0.0.0")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

# Technical timezone used while decoding the raw epoch-based integers.
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC",
)

# Read unsupported Parquet nanosecond timestamps as long integers.
spark.conf.set(
    "spark.sql.legacy.parquet.nanosAsLong",
    "true",
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print(
    "Technical decoding timezone:",
    spark.conf.get("spark.sql.session.timeZone"),
)
print(
    "Source timestamp timezone:",
    SOURCE_TIME_ZONE,
)

Spark version: 3.5.8
Spark master: spark://spark-master:7077
Technical decoding timezone: UTC
Source timestamp timezone: Europe/Berlin


## 3. Load the monthly Parquet sample

- `historical_df_raw`: the source schema exactly as Spark reads it;

In [31]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download the monthly Parquet sample first."
    )

historical_df_raw = spark.read.parquet(str(DATA_PATH))

print("Raw schema:")
historical_df_raw.printSchema()

Raw schema:
root
 |-- station_name: string (nullable = true)
 |-- xml_station_name: string (nullable = true)
 |-- eva: string (nullable = true)
 |-- train_number: string (nullable = true)
 |-- line_number: string (nullable = true)
 |-- final_destination_station: string (nullable = true)
 |-- delay_in_min: integer (nullable = true)
 |-- time: long (nullable = true)
 |-- is_canceled: boolean (nullable = true)
 |-- train_type: string (nullable = true)
 |-- train_line_ride_id: string (nullable = true)
 |-- train_line_station_num: integer (nullable = true)
 |-- arrival_planned_time: long (nullable = true)
 |-- arrival_change_time: long (nullable = true)
 |-- departure_planned_time: long (nullable = true)
 |-- departure_change_time: long (nullable = true)
 |-- id: string (nullable = true)



In [24]:
historical_df_raw.show(
    2,
    truncate=False,
    vertical=True,
)

-RECORD 0-------------------------------------------------------
 station_name              | NULL                               
 xml_station_name          | ZOB/Hauptbahnhof, Pforzheim        
 eva                       | 0940370                            
 train_number              | 33382                              
 line_number               | S6 (S                              
 final_destination_station | Bahnhof, Bad Wildbad               
 delay_in_min              | 0                                  
 time                      | 1719792000000000000                
 is_canceled               | false                              
 train_type                | Bus                                
 train_line_ride_id        | -6129702905591104469               
 train_line_station_num    | 1                                  
 arrival_planned_time      | NULL                               
 arrival_change_time       | NULL                               
 departure_planned_time  

## 4. Define the dataset-level catalog

A dataset catalog should distinguish known facts from hypotheses.

### Current grain hypothesis

> One row appears to represent one service observation for one ride at one station in the ride sequence.

This hypothesis will later be tested using:

- `id` as a candidate primary key;
- `train_line_ride_id` as a candidate natural key.

In [47]:
row_count = historical_df_raw.count()
column_count = len(historical_df_raw.columns)

# The raw Parquet value is stored in nanoseconds.
# Decode the column "time"
historical_df = historical_df_raw.withColumn(
            "time",
            F.when(
                F.col(column_name).isNotNull(),
                F.timestamp_micros(
                    (
                        F.col(column_name)
                        / F.lit(1_000)
                    ).cast("long")
                ),
            ).otherwise(
                F.lit(None).cast("timestamp")
            ),
        )

time_summary = (
    historical_df
    .agg(
        F.min("time").alias("minimum_time"),
        F.max("time").alias("maximum_time"),
        F.countDistinct(
            F.to_date("time")
        ).alias("distinct_service_dates"),
    )
    .first()
    .asDict()
)


In [49]:

dataset_catalog = {
    "dataset_name": "deutsche_bahn_historical_delays",
    "display_name": "Deutsche Bahn Historical Delay Data",
    "geographic_scope": "Germany",
    "source_platform": "Hugging Face",
    "source_repository": "piebro/deutsche-bahn-data",
    "source_url": (
        "https://huggingface.co/datasets/"
        "piebro/deutsche-bahn-data"
    ),
    "available_data_period": "From July 2024 to present",
    "coverage_period": "2024-07 to 2025-11-02 : Data for approximately the 100 largest railway stations",
    "     ": "from 2025-11-02 onward : Data for all available Deutsche Bahn stations",
    "raw_api_collection_frequency": (
    "Four times per day, approximately every six hours"
    ),
    "processed_release_frequency": "Monthly (One Parquet file per month)",
    "sample_file": DATA_PATH.name,
    "source_format": "Parquet",
    "sample_time_scope": (
        f"{time_summary['minimum_time']} "
        f"to {time_summary['maximum_time']}"
    ),
    "row_count": row_count,
    "column_count": column_count,
    "candidate_primary_key": "id",
    "internal_timezone": "Europe/Berlin (CET/CEST)",
    "catalog_status": "Verified",
}

dataset_catalog_rows = [
    (key, str(value))
    for key, value in dataset_catalog.items()
]

dataset_catalog_df = spark.createDataFrame(
    dataset_catalog_rows,
    ["catalog_attribute", "catalog_value"],
)

dataset_catalog_df.show(
    n=len(dataset_catalog_rows),
    truncate=False,
)


+----------------------------+-------------------------------------------------------------------------------+
|catalog_attribute           |catalog_value                                                                  |
+----------------------------+-------------------------------------------------------------------------------+
|dataset_name                |deutsche_bahn_historical_delays                                                |
|display_name                |Deutsche Bahn Historical Delay Data                                            |
|geographic_scope            |Germany                                                                        |
|source_platform             |Hugging Face                                                                   |
|source_repository           |piebro/deutsche-bahn-data                                                      |
|source_url                  |https://huggingface.co/datasets/piebro/deutsche-bahn-data                      |
|

## 5. Define the business column catalog

Each field receives a preliminary:

- description;
- unit or format;
- verification status.

> **Timezone:** All timestamps are Deutsche Bahn local timestamps in
> `Europe/Berlin` (CET/CEST). The source processing does not convert them to UTC.

In [29]:
column_metadata = {
    "station_name": {
        "description": (
            "Station name resolved from the EVA-to-station-name "
            "mapping used during monthly processing. It may be null "
            "when the EVA number is not present in that mapping."
        ),
        "unit_or_format": "string, nullable",
        "verification_status": (
            "Verified from monthly processing script"
        ),
    },
    "xml_station_name": {
        "description": (
            "Station name taken from the root `station` attribute "
            "of the Deutsche Bahn Timetables XML response."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "Verified from monthly processing script"
        ),
    },
    "eva": {
        "description": (
            "EVA station number: the Deutsche Bahn station "
            "identifier used in Timetables API requests."
        ),
        "unit_or_format": "string identifier",
        "verification_status": (
            "Verified from repository schema; validate DB/VBB mapping"
        ),
    },
    "train_number": {
        "description": (
            "Raw train number (`tl.n`, Zugnummer) identifying a "
            "specific train run, for example `123` or `12603`. "
            "Combine with `train_type` for labels such as `ICE 123`."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "line_number": {
        "description": (
            "Raw route or line number from `ar.l` or `dp.l`. It is "
            "non-unique across runs and is commonly null for "
            "long-distance trains such as ICE, IC, and EC."
        ),
        "unit_or_format": "string, nullable",
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "final_destination_station": {
        "description": (
            "Final destination of the train."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "Verified from monthly processing script"
        ),
    },
    "delay_in_min": {
        "description": (
            "Delay in minutes."
        ),
        "unit_or_format": "integer minutes",
        "verification_status": (
            "Formula verified from processing script; validate values"
        ),
    },
    "time": {
        "description": (
            "Actual arrival or departure time."
        ),
        "unit_or_format": (
            "timestamp in Europe/Berlin local time (CET/CEST), stored unit is nanoseconds"
        ),
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "is_canceled": {
        "description": (
            "Indicates whether the train stop was canceled. The value "
            "is derived from the presence of arrival or departure "
            "cancellation timestamps (`ar.clt` or `dp.clt`)."
        ),
        "unit_or_format": "boolean",
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "train_type": {
        "description": (
            "Raw train category from `tl.c`, for example ICE, IC, RE, "
            "RB, or S."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "train_line_ride_id": {
        "description": (
            "Unique Identifier for the train ride."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "Source-declared unique; test nulls and uniqueness in data not passed"
        ),
    },
    "train_line_station_num": {
        "description": (
            "Station number in the train's route."
            "Station position within the train ride, parsed from the "
            "last component of the source train-stop `id`."
        ),
        "unit_or_format": "integer",
        "verification_status": (
            "Verified from repository schema and processing script."
        ),
    },
    "arrival_planned_time": {
        "description": (
            "Planned arrival timestamp from the Timetables XML "
            "arrival attribute `ar.pt`."
        ),
        "unit_or_format": (
            "timestamp in Europe/Berlin local time (CET/CEST), stored unit is nanoseconds. "
        ),
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "arrival_change_time": {
        "description": (
            "Effective arrival timestamp. It uses the changed/actual "
            "arrival time from `ar.ct` when available and otherwise "
            "falls back to `arrival_planned_time`."
        ),
        "unit_or_format": (
            "timestamp in Europe/Berlin local time (CET/CEST), stored unit is nanoseconds."
        ),
        "verification_status": (
            "Verified from monthly processing script"
        ),
    },
    "departure_planned_time": {
        "description": (
            "Planned departure timestamp from the Timetables XML "
            "departure attribute `dp.pt`."
        ),
        "unit_or_format": (
            "timestamp in Europe/Berlin local time (CET/CEST), stored unit is nanoseconds."
        ),
        "verification_status": (
            "Verified from repository schema and processing script"
        ),
    },
    "departure_change_time": {
        "description": (
            "Effective departure timestamp. It uses the changed/actual "
            "departure time from `dp.ct` when available and otherwise "
            "falls back to `departure_planned_time`."
        ),
        "unit_or_format": (
            "timestamp in Europe/Berlin local time (CET/CEST), stored unit is nanoseconds."
        ),
        "verification_status": (
            "Verified from monthly processing script"
        ),
    },
    "id": {
        "description": (
            "Unique identifier for the train stop."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "Verified from repository schema, Source-declared unique; test nulls and uniqueness in data"
        ),
    },
}


## 6. Generate the combined technical and business catalog

This merges:

1. The schema Spark actually read.
2. The preliminary business descriptions.
3. Structural information such as field order, type, and nullability.

Detailed null percentages and distributions are deliberately deferred to the profiling notebook.

In [16]:
#%pip install pandas

In [51]:
catalog_rows = []

for position, field in enumerate(
    historical_df_raw.schema.fields,
    start=1,
):
    metadata = column_metadata.get(
        field.name,
        {
            "description": "Not documented yet.",
            "unit_or_format": "unknown",
            "verification_status": "Not reviewed",
        },
    )

    catalog_rows.append(
        (
            position,
            field.name,
            field.dataType.simpleString(),
            field.nullable,
            metadata["description"],
            metadata["unit_or_format"],
            metadata["verification_status"],
        )
    )

column_catalog_df = spark.createDataFrame(
    catalog_rows,
    '''
    ordinal_position int,
    column_name string,
    spark_data_type string,
    nullable boolean,
    description string,
    unit_or_format string,
    verification_status string
    ''',
)

In [52]:
import pandas as pd
from IPython.display import display

column_catalog_pd = (
    column_catalog_df
    .orderBy("ordinal_position")
    .toPandas()
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

styled_catalog = (
    column_catalog_pd.style
    .hide(axis="index")
    .set_properties(
        subset=["description", "verification_status"],
        **{
            "white-space": "normal",
            "text-align": "left",
            "min-width": "280px",
            "vertical-align": "top",
        }
    )
    .set_properties(
        **{
            "white-space": "normal",
            "vertical-align": "top",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            }
        ]
    )
)

display(styled_catalog)

ordinal_position,column_name,spark_data_type,nullable,description,unit_or_format,verification_status
1,station_name,string,True,Station name resolved from the EVA-to-station-name mapping used during monthly processing. It may be null when the EVA number is not present in that mapping.,"string, nullable",Verified from monthly processing script
2,xml_station_name,string,True,Station name taken from the root `station` attribute of the Deutsche Bahn Timetables XML response.,string,Verified from monthly processing script
3,eva,string,True,EVA station number: the Deutsche Bahn station identifier used in Timetables API requests.,string identifier,Verified from repository schema; validate DB/VBB mapping
4,train_number,string,True,"Raw train number (`tl.n`, Zugnummer) identifying a specific train run, for example `123` or `12603`. Combine with `train_type` for labels such as `ICE 123`.",string,Verified from repository schema and processing script
5,line_number,string,True,"Raw route or line number from `ar.l` or `dp.l`. It is non-unique across runs and is commonly null for long-distance trains such as ICE, IC, and EC.","string, nullable",Verified from repository schema and processing script
6,final_destination_station,string,True,Final destination of the train.,string,Verified from monthly processing script
7,delay_in_min,int,True,Delay in minutes.,integer minutes,Formula verified from processing script; validate values
8,time,bigint,True,Actual arrival or departure time.,"timestamp in Europe/Berlin local time (CET/CEST), stored unit is nanoseconds",Verified from repository schema and processing script
9,is_canceled,boolean,True,Indicates whether the train stop was canceled. The value is derived from the presence of arrival or departure cancellation timestamps (`ar.clt` or `dp.clt`).,boolean,Verified from repository schema and processing script
10,train_type,string,True,"Raw train category from `tl.c`, for example ICE, IC, RE, RB, or S.",string,Verified from repository schema and processing script


## 7. Check catalog coverage

A catalog should not silently leave physical columns undocumented.

This check reports:

- physical fields missing from the metadata dictionary;
- metadata entries that do not exist in the loaded schema;
- overall catalog coverage.

In [22]:
physical_columns = set(historical_df.columns)
documented_columns = set(column_metadata.keys())

missing_documentation = sorted(
    physical_columns - documented_columns
)

metadata_without_column = sorted(
    documented_columns - physical_columns
)

print(
    "Physical columns without documentation:",
    missing_documentation,
)

print(
    "Metadata entries without a physical column:",
    metadata_without_column,
)

catalog_coverage_pct = (
    100.0
    * len(physical_columns & documented_columns)
    / len(physical_columns)
)

print(
    f"Catalog coverage: {catalog_coverage_pct:.1f}%"
)

Physical columns without documentation: []
Metadata entries without a physical column: []
Catalog coverage: 100.0%


## 8. Test candidate keys and the grain hypothesis

Cataloging records the expected grain, but the hypothesis must be tested.

This notebook performs lightweight checks for:

- null `id` values;
- duplicate `id` groups;
- duplicate groups for the candidate natural key.

A detailed duplicate investigation belongs in the profiling and quality notebooks.

In [26]:
null_id_count = historical_df_raw.filter(
    F.col("id").isNull()
).count()

duplicate_id_groups_df = (
    historical_df_raw
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_id_group_count = (
    duplicate_id_groups_df.count()
)

candidate_natural_key = [
    "train_line_ride_id"
]

duplicate_natural_key_groups_df = (
    historical_df
    .groupBy(*candidate_natural_key)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_natural_key_group_count = (
    duplicate_natural_key_groups_df.count()
)

key_assessment_rows = [
    (
        "id",
        "candidate_primary_key",
        null_id_count,
        duplicate_id_group_count,
        (
            null_id_count == 0
            and duplicate_id_group_count == 0
        ),
    ),
    (
        " + ".join(candidate_natural_key),
        "candidate_natural_key",
        None,
        duplicate_natural_key_group_count,
        duplicate_natural_key_group_count == 0,
    ),
]

key_assessment_df = spark.createDataFrame(
    key_assessment_rows,
    '''
    key_definition string,
    key_type string,
    null_key_count long,
    duplicate_group_count long,
    passes_initial_check boolean
    ''',
)

key_assessment_df.show(truncate=False)

+------------------+---------------------+--------------+---------------------+--------------------+
|key_definition    |key_type             |null_key_count|duplicate_group_count|passes_initial_check|
+------------------+---------------------+--------------+---------------------+--------------------+
|id                |candidate_primary_key|0             |0                    |true                |
|train_line_ride_id|candidate_natural_key|NULL          |57038                |false               |
+------------------+---------------------+--------------+---------------------+--------------------+



## 10. Export the data catalog

In [53]:
catalog_outputs = {
    "dataset_catalog": dataset_catalog_df,
    "column_catalog": column_catalog_df,
    "key_assessment": key_assessment_df,
}

for output_name, output_df in catalog_outputs.items():
    parquet_path = str(
        CATALOG_BASE_PATH
        / f"{output_name}_parquet"
    )

    csv_path = str(
        CATALOG_BASE_PATH
        / f"{output_name}_csv"
    )

    (
        output_df.write
        .mode("overwrite")
        .parquet(parquet_path)
    )

    (
        output_df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(csv_path)
    )

    print(f"Saved {output_name}:")
    print("  Parquet:", parquet_path)
    print("  CSV:", csv_path)

Saved dataset_catalog:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/dataset_catalog_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/dataset_catalog_csv
Saved column_catalog:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/column_catalog_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/column_catalog_csv
Saved key_assessment:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/key_assessment_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/key_assessment_csv
